# Presto first run on real project data

**Scope.** This notebook shows that the pinned, pretrained [Presto](https://github.com/nasaharvest/presto) model loads
and that its encoder runs on inputs built from this project's own data: ten Malawi 2023 field polygons, their
Sentinel-2 Collection 1 chips, and one monthly reflectance series per field.

**It proves that the model runs. It does not show that Presto predicts yield.** Nothing is trained or evaluated here;
yields appear only as labels in tables.

Before running (see README, *Presto first run*): `python -m pip install -e ".[dev,presto]"`, `python scripts/get_presto.py`,
and the ten pilot chips from `scripts/download_sentinel2.py`. Restart the kernel and run all cells.

In [1]:
from datetime import date

import numpy as np
import pandas as pd
import torch

from cropyield import ROOT
from cropyield.data.cohorts import load_cohort
from cropyield.features.presto_inputs import NORMED_BANDS, presto_inputs
from cropyield.models import presto_weights as weights
from cropyield.models.presto import EMBEDDING_SIZE, encode, load_presto, to_tensors
from cropyield.satellite.chips import read_chip
from cropyield.satellite.monthly import monthly_series

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
print("repository:", ROOT)
print("torch", torch.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

repository: /home/user/Data_science_for_sustainable_development/project/DSC-291-Final-Project
torch 2.14.1+cu130 | numpy 2.5.3 | pandas 3.0.5


## 1. The ten records

`load_cohort` reads the screened audit CSV for the Malawi 2023 cohort in file order and gives every row a stable
`obs_id` (`dataset:source_id`). The pilot is its first ten rows, the same ten listed in
`configs/sentinel2_windows_malawi_2023_pilot.csv`. `read_chip` reads each record folder the downloader wrote.

In [2]:
cohort = load_cohort("malawi_2023")
pilot = cohort.head(10).set_index("obs_id")

chips = {}  # keep, per record, the complete chip with the most scenes among those downloaded for the season
for run in sorted((ROOT / "data" / "imagery" / "sentinel2").glob("run-*")):
    for record_dir in sorted((run / "records").glob("malawi_2023__*")):
        if (record_dir / "record.json").exists():
            chip = read_chip(record_dir)
            if chip.obs_id in pilot.index and chip.start_date is not None and chip.end_date is not None:
                season = pilot.loc[chip.obs_id]
                if str(chip.start_date) > season["season_start"] or str(chip.end_date) < season["season_end"]:
                    continue  # downloaded for another window
                better = (chip.status == "complete", len(chip.scenes))
                if chip.obs_id not in chips or better > (chips[chip.obs_id].status == "complete", len(chips[chip.obs_id].scenes)):
                    chips[chip.obs_id] = chip
missing = [obs_id for obs_id in pilot.index if obs_id not in chips]
assert not missing, f"run the downloader first; no chip folder for {missing}"

records = pd.DataFrame({
    "source_row": pilot["source_row"],
    "yield_kg_ha": pilot["yield_kg_ha"],
    "latitude": pilot["latitude"].round(4),
    "longitude": pilot["longitude"].round(4),
    "field_area_ha": pilot["field_area_ha"].astype(float).round(2),
    "season": pilot["season_start"] + " .. " + pilot["season_end"],
    "aoi": [chips[i].aoi_type for i in pilot.index],
    "touched_pixels": [int(chips[i].aoi.sum()) for i in pilot.index],
    "interior_pixels": [int(chips[i].aoi_interior.sum()) for i in pilot.index],
    "status": [chips[i].status for i in pilot.index],
    "scenes": [len(chips[i].scenes) for i in pilot.index],
})
print(f"{len(cohort)} rows in the cohort, {len(pilot)} in the pilot, {len(chips)} chips read")
records

260 rows in the cohort, 10 in the pilot, 10 chips read


,source_row,yield_kg_ha,latitude,longitude,field_area_ha,season,aoi,touched_pixels,interior_pixels,status,scenes
obs_id,,,,,,,,,,,
malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,1,2950.0,-13.3171,33.3837,0.80,2022-10-01 .. 2023-06-30,field_polygon,109,79,complete,5
malawi_2023:626290f6-99b2-4a44-b233-8df0e13d3e63,2,2300.0,-13.3220,33.3684,0.53,2022-10-01 .. 2023-06-30,field_polygon,73,52,complete,4
malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,3,2620.0,-13.2915,33.4005,1.19,2022-10-01 .. 2023-06-30,field_polygon,147,118,complete,5
malawi_2023:1d60c29e-ee3a-4639-9e07-0239523dfc62,4,3310.0,-13.2900,33.4008,0.83,2022-10-01 .. 2023-06-30,field_polygon,112,84,complete,5
malawi_2023:6fa8a978-4710-469b-b593-e100746035dd,5,4510.0,-13.3022,33.4029,0.66,2022-10-01 .. 2023-06-30,field_polygon,88,64,complete,5
malawi_2023:79ff56b9-ea86-438b-ad8b-6859295353db,6,2880.0,-13.2976,33.4029,0.51,2022-10-01 .. 2023-06-30,field_polygon,69,50,complete,6
malawi_2023:c3ca5a86-2c24-4c45-a762-9c766fb0fb1d,7,3450.0,-13.2953,33.3935,0.57,2022-10-01 .. 2023-06-30,field_polygon,75,58,complete,6
malawi_2023:4591d3d5-8258-4a9e-a8db-ad1024acb672,8,780.0,-13.2954,33.4044,0.82,2022-10-01 .. 2023-06-30,field_polygon,104,80,complete,7
malawi_2023:711f3b9b-a6f7-4876-aeaf-5062b5ac688f,9,2930.0,-13.2949,33.3944,0.46,2022-10-01 .. 2023-06-30,field_polygon,73,45,complete,6


## 2. One record in detail: location, season window and accepted scenes

Each accepted scene passed the downloader's check of at least 80 % clear, valid pixels over the field. Reflectance in
the chips is unitless bottom-of-atmosphere reflectance with the Collection 1 offset already applied
(`docs/DECISIONS.md`, 26 October 2026).

In [3]:
first_id = pilot.index[0]
first = chips[first_id]
row = pilot.loc[first_id]
season = (date.fromisoformat(row["season_start"]), date.fromisoformat(row["season_end"]))
print("obs_id        ", first.obs_id)
print("source row    ", first.source_row, "| yield", first.yield_kg_ha, "kg/ha")
print("location      ", f"lat {first.latitude:.5f}, lon {first.longitude:.5f}", "| grid", first.crs, first.aoi.shape)
print("season window ", season[0], "..", season[1], "| downloader window", first.start_date, "..", first.end_date, f"({first.window_basis})")
print("field         ", first.aoi_type, "|", int(first.aoi.sum()), "pixels touched by the polygon,", int(first.aoi_interior.sum()), "with their centre inside | entirely inside chip:", first.entire_aoi_inside_chip)
pd.DataFrame({
    "item_id": [s.item_id for s in first.scenes],
    "date": [s.date for s in first.scenes],
    "tile_cloud_%": [round(s.tile_cloud_cover, 1) for s in first.scenes],
    "clear_fraction_aoi": [round(s.clear_fraction_aoi, 3) for s in first.scenes],
    "valid_interior_pixels": [int((first.aoi_interior & s.valid).sum()) for s in first.scenes],
    "reflectance_min": [float(np.nanmin(s.reflectance)) for s in first.scenes],
    "reflectance_max": [float(np.nanmax(s.reflectance)) for s in first.scenes],
})

obs_id         malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71
source row     1 | yield 2950.0 kg/ha
location       lat -13.31707, lon 33.38368 | grid EPSG:32736 (64, 64)
season window  2022-10-01 .. 2023-06-30 | downloader window 2022-10-01 .. 2023-06-30 (user_override)
field          field_polygon | 109 pixels touched by the polygon, 79 with their centre inside | entirely inside chip: True


,item_id,date,tile_cloud_%,clear_fraction_aoi,valid_interior_pixels,reflectance_min,reflectance_max
0,S2B_T36LWL_20230113T080204_L2A,2023-01-13,68.1,1.0,79,0.0542,0.572031
1,S2B_T36LWL_20230307T081156_L2A,2023-03-07,25.3,1.0,79,0.0291,0.440488
2,S2B_T36LWL_20230426T081153_L2A,2023-04-26,3.4,1.0,79,0.0306,0.370719
3,S2B_T36LWL_20230506T081154_L2A,2023-05-06,0.1,1.0,79,0.0298,0.370925
4,S2A_T36LWL_20230630T080627_L2A,2023-06-30,0.0,1.0,79,0.0360,0.353238


## 3. Monthly band values and missing months

`monthly_series` gives one row per calendar month of the season window: the median reflectance over the pixels whose centres
lie inside the field polygon and that are valid in that month's scene (the downloader's all-touched mask adds boundary pixels). A month without an accepted scene is NaN and is listed as missing. The same
series, through its named bands, feeds the NDVI/GCVI ridge baseline later.

In [4]:
series = monthly_series(first, *season)
monthly = pd.DataFrame(series.values, index=series.months, columns=series.bands)
monthly["NDVI"] = (series.band("B08") - series.band("B04")) / (series.band("B08") + series.band("B04"))
monthly["GCVI"] = series.band("B08") / series.band("B03") - 1
monthly["valid_pixels"] = series.n_pixels
monthly["scene"] = [i or "" for i in series.item_ids]
print("pixel rule:", series.pixel_rule, "| missing months:", [m for m, miss in zip(series.months, series.missing) if miss])
monthly.round(3)

pixel rule: interior | missing months: ['2022-10', '2022-11', '2022-12', '2023-02']


,B02,B03,B04,B05,B06,B07,B08,B8A,B11,B12,NDVI,GCVI,valid_pixels,scene
2022-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,
2022-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,
2022-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,
2023-01,0.114,0.149,0.180,0.220,0.278,0.313,0.284,0.333,0.416,0.390,0.224,0.903,79,S2B_T36LWL_20230113T080204_L2A
2023-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,
2023-03,0.042,0.066,0.061,0.108,0.224,0.265,0.257,0.280,0.207,0.129,0.618,2.870,79,S2B_T36LWL_20230307T081156_L2A
2023-04,0.090,0.118,0.118,0.173,0.264,0.290,0.299,0.315,0.273,0.189,0.434,1.535,79,S2B_T36LWL_20230426T081153_L2A
2023-05,0.050,0.075,0.086,0.141,0.214,0.245,0.254,0.269,0.255,0.178,0.493,2.372,79,S2B_T36LWL_20230506T081154_L2A
2023-06,0.060,0.086,0.119,0.159,0.201,0.223,0.236,0.255,0.286,0.209,0.329,1.751,79,S2A_T36LWL_20230630T080627_L2A


## 4. Presto inputs from the series

`presto_inputs` places the ten Sentinel-2 bands into Presto's 17-band layout, computes NDVI the way Presto does, and
masks everything we do not have: Sentinel-1, ERA5 and SRTM at every timestep, and every band of a missing month.
Presto normalises Sentinel-2 digital numbers by dividing by 10 000; our reflectance is already on that scale, so the
value Presto sees equals the reflectance in the table above (checked below).

In [5]:
inputs = presto_inputs(series, row["latitude"], row["longitude"])
print("x            ", inputs.x.shape, inputs.x.dtype, "(batch, months, 17 bands)")
print("mask         ", inputs.mask.shape, "| masked entries:", int(inputs.mask.sum()), "of", inputs.mask.size)
print("dynamic_world", inputs.dynamic_world.shape, "| unique values:", np.unique(inputs.dynamic_world).tolist(), "(9 = unknown, ignored)")
print("latlons      ", inputs.latlons.tolist(), "| first month index (0 = January):", inputs.month)
scale_check = np.allclose(inputs.x[0, ~series.missing, NORMED_BANDS.index("B8")], series.band("B08")[~series.missing])
print("x[B8] equals reflectance B08 for present months:", scale_check)
per_month = pd.DataFrame({
    "unmasked_bands": (inputs.mask[0] == 0).sum(axis=1).astype(int),
    "x_B4_red": inputs.x[0, :, NORMED_BANDS.index("B4")].round(3),
    "x_B8_nir": inputs.x[0, :, NORMED_BANDS.index("B8")].round(3),
    "x_NDVI": inputs.x[0, :, NORMED_BANDS.index("NDVI")].round(3),
    "masked_bands": [", ".join(b for b, m in zip(NORMED_BANDS, mask_row) if m == 1) for mask_row in inputs.mask[0]],
}, index=inputs.months)
per_month

x             (1, 9, 17) float32 (batch, months, 17 bands)
mask          (1, 9, 17) | masked entries: 98 of 153
dynamic_world (1, 9) | unique values: [9] (9 = unknown, ignored)
latlons       [[-13.317068099975586, 33.3836784362793]] | first month index (0 = January): 9
x[B8] equals reflectance B08 for present months: True


,unmasked_bands,x_B4_red,x_B8_nir,x_NDVI,masked_bands
2022-10,0,0.000,0.000,0.000,"VV, VH, B2, B3, B4, B5, B6, B7, B8, B8A, B11, ..."
2022-11,0,0.000,0.000,0.000,"VV, VH, B2, B3, B4, B5, B6, B7, B8, B8A, B11, ..."
2022-12,0,0.000,0.000,0.000,"VV, VH, B2, B3, B4, B5, B6, B7, B8, B8A, B11, ..."
2023-01,11,0.180,0.284,0.224,"VV, VH, temperature_2m, total_precipitation, e..."
2023-02,0,0.000,0.000,0.000,"VV, VH, B2, B3, B4, B5, B6, B7, B8, B8A, B11, ..."
2023-03,11,0.061,0.257,0.618,"VV, VH, temperature_2m, total_precipitation, e..."
2023-04,11,0.118,0.299,0.434,"VV, VH, temperature_2m, total_precipitation, e..."
2023-05,11,0.086,0.254,0.493,"VV, VH, temperature_2m, total_precipitation, e..."
2023-06,11,0.119,0.236,0.329,"VV, VH, temperature_2m, total_precipitation, e..."


## 5. Load the pinned checkpoint

The weights are `data/default_model.pt` from nasaharvest/presto at the pinned commit, downloaded by
`scripts/get_presto.py`; `load_presto` refuses a file whose size or SHA-256 differs and loads the state dict strictly
into the vendored model code (`src/cropyield/models/presto_single_file.py`, same commit).

In [6]:
print("commit ", weights.PRESTO_COMMIT)
print("url    ", weights.PRESTO_WEIGHTS_URL)
print("path   ", weights.PRESTO_WEIGHTS_PATH.relative_to(ROOT), "|", weights.PRESTO_WEIGHTS_PATH.stat().st_size, "bytes")
print("sha256 ", weights.sha256(weights.PRESTO_WEIGHTS_PATH), "| matches pin:", weights.weights_verified())
model = load_presto()
n_encoder = sum(p.numel() for p in model.encoder.parameters())
n_total = sum(p.numel() for p in model.parameters())
print(f"loaded strictly | encoder parameters {n_encoder:,} | encoder + decoder {n_total:,} | eval mode: {not model.training}")

commit  11e207a668a34336ced1d8e492a1bd5849b96c4a
url     https://raw.githubusercontent.com/nasaharvest/presto/11e207a668a34336ced1d8e492a1bd5849b96c4a/data/default_model.pt
path    data/models/presto/default_model.pt | 3321716 bytes
sha256  559c9d76f525bdf2dd3774da8b1e7f1510769b73baacac12e9e4d8d5c67fbe65 | matches pin: True
loaded strictly | encoder parameters 404,160 | encoder + decoder 822,682 | eval mode: True


## 6. Run the encoder on all ten records

One record at a time: Presto's encoder requires every item of a batch to have the same number of masked tokens, and
the number of missing months differs between records. `encode` returns Presto's `eval_task` output: the final LayerNorm of the mean over the kept tokens (the
unmasked band-group tokens of each month plus the latitude/longitude token), a 128-dimensional embedding per field.

In [7]:
rows, embeddings = [], []
for obs_id in pilot.index:
    chip, r = chips[obs_id], pilot.loc[obs_id]
    s = monthly_series(chip, date.fromisoformat(r["season_start"]), date.fromisoformat(r["season_end"]))
    inp = presto_inputs(s, r["latitude"], r["longitude"])
    emb = encode(model, inp)
    embeddings.append(emb)
    rows.append({"months_with_data": int((~s.missing).sum()), "months_missing": int(s.missing.sum()),
                 "x_shape": tuple(inp.x.shape), "embedding_shape": emb.shape,
                 "finite": bool(np.isfinite(emb).all()), "norm": round(float(np.linalg.norm(emb)), 3),
                 "yield_kg_ha": r["yield_kg_ha"]})
embeddings = np.stack(embeddings)
summary = pd.DataFrame(rows, index=pilot.index)
print("embeddings", embeddings.shape, "| all finite:", bool(np.isfinite(embeddings).all()),
      "| all records produced an embedding:", len(embeddings) == len(pilot))
summary

embeddings (10, 128) | all finite: True | all records produced an embedding: True


,months_with_data,months_missing,x_shape,embedding_shape,finite,norm,yield_kg_ha
obs_id,,,,,,,
malawi_2023:506ae8b2-d085-4eeb-b807-765e03925c71,5,4,"(1, 9, 17)","(128,)",True,12.929,2950.0
malawi_2023:626290f6-99b2-4a44-b233-8df0e13d3e63,4,5,"(1, 9, 17)","(128,)",True,12.828,2300.0
malawi_2023:10738537-aded-4abe-bc18-e3d8ce52f7f7,5,4,"(1, 9, 17)","(128,)",True,12.954,2620.0
malawi_2023:1d60c29e-ee3a-4639-9e07-0239523dfc62,5,4,"(1, 9, 17)","(128,)",True,12.778,3310.0
malawi_2023:6fa8a978-4710-469b-b593-e100746035dd,5,4,"(1, 9, 17)","(128,)",True,12.897,4510.0
malawi_2023:79ff56b9-ea86-438b-ad8b-6859295353db,6,3,"(1, 9, 17)","(128,)",True,12.950,2880.0
malawi_2023:c3ca5a86-2c24-4c45-a762-9c766fb0fb1d,6,3,"(1, 9, 17)","(128,)",True,12.936,3450.0
malawi_2023:4591d3d5-8258-4a9e-a8db-ad1024acb672,7,2,"(1, 9, 17)","(128,)",True,12.927,780.0
malawi_2023:711f3b9b-a6f7-4876-aeaf-5062b5ac688f,6,3,"(1, 9, 17)","(128,)",True,13.047,2930.0


In [8]:
print("first 8 dimensions of each embedding:")
print(pd.DataFrame(embeddings[:, :8], index=[i.split(":")[1][:8] for i in pilot.index]).round(3))
unit = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
print("\ncosine similarity between the ten field embeddings (1 on the diagonal):")
print(pd.DataFrame(unit @ unit.T, index=range(1, 11), columns=range(1, 11)).round(2))

first 8 dimensions of each embedding:
              0      1      2      3      4      5      6      7
506ae8b2 -0.872 -4.672  2.101 -0.102 -0.318  1.612 -1.136  0.483
626290f6 -0.699 -4.596  1.982 -0.213 -0.215  1.372 -1.044  0.563
10738537 -0.949 -4.733  2.095 -0.345 -0.450  1.598 -1.254  0.498
1d60c29e -0.996 -4.742  2.111 -0.487 -0.492  1.038 -1.150  0.322
6fa8a978 -1.345 -5.008  1.807 -0.326 -0.524  1.374 -1.247  0.515
79ff56b9 -1.284 -4.822  1.996 -0.411 -0.351  1.380 -1.354  0.440
c3ca5a86 -1.206 -4.875  1.910 -0.271 -0.442  1.377 -1.228  0.510
4591d3d5 -1.170 -4.949  2.041 -0.297 -0.408  1.534 -1.123  0.551
711f3b9b -1.338 -4.874  1.934 -0.235 -0.356  1.593 -1.332  0.584
ea2ca6d4 -1.177 -4.759  2.108 -0.305 -0.280  1.587 -1.386  0.407

cosine similarity between the ten field embeddings (1 on the diagonal):
      1     2     3     4     5     6     7     8     9     10
1   1.00  0.97  0.99  0.97  0.97  0.98  0.97  0.98  0.98  0.99
2   0.97  1.00  0.97  0.99  0.98  0.98  0.98  0.

## 7. The official regression head attaches (untrained)

`construct_finetuning_model(num_outputs=1, regression=True)` is the head the research plan names. Its output here comes
from random initial weights and means nothing; the cell only shows that the shapes line up for later fine-tuning.

In [9]:
head_model = model.construct_finetuning_model(num_outputs=1, regression=True).eval()
with torch.no_grad():
    prediction = head_model(**to_tensors(inputs))
print("untrained head output", tuple(prediction.shape), "| finite:", bool(torch.isfinite(prediction).all()))
print("trainable parameters in the fine-tuning model:", f"{sum(p.numel() for p in head_model.parameters() if p.requires_grad):,}")

untrained head output (1, 1) | finite: True
trainable parameters in the fine-tuning model: 402,369


## What this shows, and what it does not

Shown: the pinned checkpoint loads strictly into the pinned model code; real chips become a monthly series with the
expected missing months; Presto's inputs have the documented shapes and scale; the frozen encoder produces a finite
128-dimensional embedding for every pilot field; the regression head attaches.

Not shown: that these embeddings carry yield information. That needs the frozen-embedding baseline under the frozen
spatial splits, which is week-1 work in `docs/ROADMAP.md`.